# Event display examples

This notebook demonstrates the three `track_vis` plot types on a synthetic event, then on a real MC event read straight from a ROOT ntuple:

- `plot_tracks` — per-particle 3D trajectories, colour-coded by time
- `plot_truth_vertices` — MC truth vertex markers, grouped by species
- `plot_voxels` — deposited energy binned into cubes

All three functions take columnar arrays matching the SNO+ `imaging_reco` ntuple schema: a flat `pdg` array plus jagged (ragged) `pos_x`/`pos_y`/`pos_z`/`time`/`deposited_energy` arrays (outer index = track, inner index = step).

In [ ]:
import awkward as ak
import numpy as np
import plotly.graph_objects as go

from track_vis import load_event, plot_tracks, plot_truth_vertices, plot_voxels

rng = np.random.default_rng(42)

## Build synthetic tracks

Each track is a helical/curved random walk from a common vertex. `pdg` is a flat list (one entry per track); `pos_x`, `pos_y`, `pos_z`, `time`, `deposited_energy` are jagged, one list of steps per track.

In [ ]:
def make_track(start, direction, n_steps, step_len, t0, dedx, curl=0.0):
    """Fake a particle trajectory: a straight-ish path that curls by `curl` rad/step."""
    direction = direction / np.linalg.norm(direction)
    perp = np.cross(direction, [0, 0, 1])
    if np.linalg.norm(perp) < 1e-6:
        perp = np.cross(direction, [0, 1, 0])
    perp = perp / np.linalg.norm(perp)

    positions = [start]
    for i in range(1, n_steps):
        theta = curl * i
        bend = perp * np.sin(theta) * step_len
        step = direction * step_len + bend
        positions.append(positions[-1] + step)
    positions = np.array(positions)

    times = t0 + np.arange(n_steps) * (step_len / 200.0)
    dep = np.full(n_steps, dedx) * rng.uniform(0.7, 1.3, n_steps)

    return positions, times, dep


vertex_pos = np.array([0.0, 0.0, 0.0])
t0 = 0.0

particles = [
    (11, np.array([1, 0.3, 0.2]), 40, 15.0, t0, 0.3, 0.15),  # electron, curls in field
    (2212, np.array([-0.5, 1, 0.1]), 25, 20.0, t0 + 0.5, 1.2, 0.0),  # proton, minimal curl
    (22, np.array([-0.2, -0.8, 0.9]), 30, 25.0, t0 + 0.2, 0.05, 0.0),  # gamma, light dep.
]

pdg = []
pos_x, pos_y, pos_z, time, deposited_energy = [], [], [], [], []
for pdg_code, direction, n_steps, step_len, start_t, dedx, curl in particles:
    positions, times, dep = make_track(vertex_pos, direction, n_steps, step_len, start_t, dedx, curl)
    pdg.append(pdg_code)
    pos_x.append(positions[:, 0].tolist())
    pos_y.append(positions[:, 1].tolist())
    pos_z.append(positions[:, 2].tolist())
    time.append(times.tolist())
    deposited_energy.append(dep.tolist())

synthetic_tracks = {
    "pdg": np.array(pdg),
    "pos_x": ak.Array(pos_x),
    "pos_y": ak.Array(pos_y),
    "pos_z": ak.Array(pos_z),
    "time": ak.Array(time),
    "deposited_energy": ak.Array(deposited_energy),
}
synthetic_tracks

## Build synthetic truth vertices

`plot_truth_vertices` expects flat arrays `pdg`, `x`, `y`, `z`, `t`, `energy` — one entry per vertex.

In [ ]:
vertex_2 = vertex_pos + np.array([5, -3, 2])

synthetic_vertices = {
    "pdg": np.array([11, 2212]),
    "x": np.array([vertex_pos[0], vertex_2[0]]),
    "y": np.array([vertex_pos[1], vertex_2[1]]),
    "z": np.array([vertex_pos[2], vertex_2[2]]),
    "t": np.array([0.0, 1.1]),
    "energy": np.array([2.5, 0.8]),
}
synthetic_vertices

## Assemble the synthetic figure

`plot_voxels` bins every step's deposited energy into cubes, giving a coarse calorimetric view alongside the sharp track lines and vertex markers.

In [ ]:
all_times = ak.to_numpy(ak.flatten(synthetic_tracks["time"]))
t_min, t_max = float(all_times.min()), float(all_times.max())

traces = plot_tracks(**synthetic_tracks, t_min=t_min, t_max=t_max)
traces += plot_truth_vertices(**synthetic_vertices, t_min=t_min, t_max=t_max)
traces += plot_voxels(
    pos_x=synthetic_tracks["pos_x"],
    pos_y=synthetic_tracks["pos_y"],
    pos_z=synthetic_tracks["pos_z"],
    deposited_energy=synthetic_tracks["deposited_energy"],
    voxel_size=10.0,
)

fig = go.Figure(data=traces)
fig.update_layout(
    scene=dict(
        xaxis_title="x (mm)", yaxis_title="y (mm)", zaxis_title="z (mm)", aspectmode="data"
    ),
    height=700,
    legend=dict(itemsizing="constant"),
    title="Synthetic event: e- / p / gamma from a common vertex",
)
fig.show()

## Real data: reading an event from a ROOT ntuple

`load_event` opens a SNO+ `imaging_reco` ntuple with `uproot` and returns `(tracks, vertices)` kwargs dicts ready to unpack straight into `plot_tracks`/`plot_voxels`/`plot_truth_vertices` — no manual column wrangling needed.

This reads event 1 from a 2.5 MeV uniform-fiducial electron MC sample, which has a handful of bremsstrahlung/ionization secondaries in addition to the primary electron.

In [ ]:
path = "/data/snoplus2/degraw/imaging_reco/mc/electron_uniform_fiducial/root/job_0.ntuple.root"
real_tracks, real_vertices = load_event(path, index=1)

print(f"{len(real_tracks['pdg'])} tracks, pdgs={ak.to_list(real_tracks['pdg'])}")

In [ ]:
real_times = ak.to_numpy(ak.flatten(real_tracks["time"]))
t_min, t_max = float(real_times.min()), float(real_times.max())

real_traces = plot_tracks(**real_tracks, t_min=t_min, t_max=t_max)
real_traces += plot_truth_vertices(**real_vertices, t_min=t_min, t_max=t_max)
real_traces += plot_voxels(
    pos_x=real_tracks["pos_x"],
    pos_y=real_tracks["pos_y"],
    pos_z=real_tracks["pos_z"],
    deposited_energy=real_tracks["deposited_energy"],
    voxel_size=50.0,
)

real_fig = go.Figure(data=real_traces)
real_fig.update_layout(
    scene=dict(
        xaxis_title="x (mm)", yaxis_title="y (mm)", zaxis_title="z (mm)", aspectmode="data"
    ),
    height=700,
    legend=dict(itemsizing="constant"),
    title="Real event: 2.5 MeV electron, uniform fiducial MC",
)
real_fig.show()